<a href="https://colab.research.google.com/github/vzavala/StatsBook/blob/master/Python/ch5_hougen_watson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Setup.** Run this cell first: it locates the data files this notebook reads. On Google
Colab it downloads them (the `Data/` and `Support/` folders) from the book's GitHub
repository; on your computer, run the notebook from the folder that contains `Data/`.

In [1]:
import os
import sys
path = ''
if 'google.colab' in sys.modules and not os.path.isdir('Data'):
    if not os.path.isdir('statsbook'):
        os.system('git clone -q --depth 1 --filter=blob:none --sparse '
                  'https://github.com/vzavala/StatsBook.git statsbook')
        os.system('git -C statsbook sparse-checkout set Python/Data Python/Support')
    path = 'statsbook/Python/'

Nonlinear estimation for the Hougen-Watson model (Victor Z and Leo Gonzalez, UW-Madison, 2024).

$$m_\omega(\theta)=\frac{\theta_0x_{2,\omega}-x_{3,\omega}/\theta_4}{1+\theta_1x_{1,\omega}+\theta_2x_{2,\omega}+\theta_3x_{3,\omega}}$$

| Slides | Python | Meaning |
|---|---|---|
| $x_{1,\omega},x_{2,\omega},x_{3,\omega}$ | `X[:, 0]`, `X[:, 1]`, `X[:, 2]` | partial pressures ($S=13$ experiments) |
| $y_\omega$ | `y` | reaction rate |
| $\theta=(\theta_0,\dots,\theta_4)$ | `theta[0]`, ..., `theta[4]` | parameters |
| $m(\theta)$, $SSE(\theta)=\frac{1}{2}\lVert\mathbf{y}-\mathbf{m}(\theta)\rVert^2$ | `m(theta, X)`, `SSE(theta)` | model and objective |

## The SSE function is highly nonlinear

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

data = np.loadtxt(path+'Data/hougenwatson.dat')
X, y = data[:, :3], data[:, -1]
S = len(y)

def m(theta, X):
    """Hougen-Watson model m(theta, x) for all rows of X."""
    x1, x2, x3 = X[:, 0], X[:, 1], X[:, 2]
    return (theta[0]*x2 - x3/theta[4]) / (1 + theta[1]*x1 + theta[2]*x2 + theta[3]*x3)

def SSE(theta):
    return 0.5 * np.sum((y - m(theta, X))**2)

$SSE(\theta)$ as we span $\theta_1,\theta_2\in[-2,2]$ (other parameters fixed at a reference value).
Note the poles where the denominator of $m$ vanishes.

In [3]:
theta_ref = np.array([1.0, 0.05, 0.02, 0.1, 2.0])
grid = np.linspace(-2, 2, 201)
SSE_map = np.zeros((len(grid), len(grid)))
for i, t2 in enumerate(grid):
    for j, t1 in enumerate(grid):
        th = theta_ref.copy()
        th[1], th[2] = t1, t2
        SSE_map[i, j] = SSE(th)
plt.imshow(np.log10(SSE_map), extent=[-2, 2, -2, 2], origin='lower', cmap='gray_r', aspect='auto')
plt.colorbar(label=r'$\log_{10} SSE(\theta)$')
plt.xlabel(r'$\theta_1$', fontsize=14)
plt.ylabel(r'$\theta_2$', fontsize=14)
plt.show()

/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40568/932353689.py:12: RuntimeWarning: divide by zero encountered in divide
  return (theta[0]*x2 - x3/theta[4]) / (1 + theta[1]*x1 + theta[2]*x2 + theta[3]*x3)
/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40568/1928643847.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Good and bad initial guesses

Solve $\min_\theta \frac{1}{2}\lVert\mathbf{y}-\mathbf{m}(\theta)\rVert^2$ with a Levenberg-Marquardt
(Newton-like) method, from $\theta=(1,\dots,1)$ and from $\theta=(-1,\dots,-1)$.
(The slides use Matlab's `fitnlm`; `scipy.optimize.least_squares` plays the same role.)
As in the book, the reported value is the sum of squared residuals (without the factor 1/2).

In [4]:
fits = {}
for label, theta0 in (("theta0 = (1,...,1)", np.ones(5)), ("theta0 = (-1,...,-1)", -np.ones(5))):
    sol = least_squares(lambda th: y - m(th, X), theta0, method='lm', max_nfev=2000)
    fits[label] = sol.x
    print(f"{label}:  theta_hat = {np.round(sol.x, 2)},  SSE = {2*SSE(sol.x):.4f}")

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, (label, th) in zip(axs, fits.items()):
    ax.plot([0, 15], [0, 15], 'k--', linewidth=1)
    ax.scatter(m(th, X), y, marker='o', color='w', edgecolor='k', s=50)
    ax.set_title(label)
    ax.set_xlabel(r'$\hat{y}$'); ax.set_ylabel('$y$')
    ax.axis([0, 15, 0, 15])
    ax.grid(True)
plt.tight_layout()
plt.show()

theta0 = (1,...,1):  theta_hat = [1.25 0.06 0.04 0.11 1.19],  SSE = 0.2989
theta0 = (-1,...,-1):  theta_hat = [-2887567.23  -183298.6    -58849.71  -652886.96    37012.98],  SSE = 11.3903


/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40568/1981136141.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
